# Python Advanced — AI Engineering Interview Prep

Covers: async/await, decorator factories, design patterns, slots, descriptors.

## 1. asyncio — Concurrency without Threads

In [ ]:
import asyncio
import time

async def fetch(url: str, delay: float) -> str:
    """Simulates an async HTTP request."""
    await asyncio.sleep(delay)
    return f"Response from {url} (after {delay}s)"

async def main_sequential():
    start = time.perf_counter()
    r1 = await fetch("api/users", 0.1)
    r2 = await fetch("api/orders", 0.1)
    print(f"Sequential: {time.perf_counter()-start:.2f}s")
    return [r1, r2]

async def main_concurrent():
    start = time.perf_counter()
    # asyncio.gather runs coroutines concurrently
    results = await asyncio.gather(
        fetch("api/users", 0.1),
        fetch("api/orders", 0.1),
        fetch("api/products", 0.1),
    )
    print(f"Concurrent: {time.perf_counter()-start:.2f}s")
    return results

results = asyncio.run(main_concurrent())
for r in results:
    print(r)

In [ ]:
# asyncio.create_task + Queue (producer-consumer pattern)
async def producer(queue: asyncio.Queue, items):
    for item in items:
        await queue.put(item)
        await asyncio.sleep(0.01)
    await queue.put(None)  # sentinel

async def consumer(queue: asyncio.Queue, name: str):
    results = []
    while True:
        item = await queue.get()
        if item is None:
            await queue.put(None)  # re-signal for other consumers
            break
        results.append(f"{name} processed {item}")
    return results

async def pipeline():
    q = asyncio.Queue(maxsize=5)
    prod = asyncio.create_task(producer(q, range(5)))
    cons = asyncio.create_task(consumer(q, "Worker-1"))
    await prod
    results = await cons
    for r in results:
        print(r)

asyncio.run(pipeline())

## 2. Decorator Factories

In [ ]:
import functools
import time

# Decorator that accepts arguments
def retry(max_attempts: int = 3, delay: float = 0.0, exceptions=(Exception,)):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(1, max_attempts + 1):
                try:
                    return func(*args, **kwargs)
                except exceptions as e:
                    if attempt == max_attempts:
                        raise
                    if delay:
                        time.sleep(delay)
        return wrapper
    return decorator

def validate_types(**type_map):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            import inspect
            sig = inspect.signature(func)
            bound = sig.bind(*args, **kwargs)
            for param, expected in type_map.items():
                if param in bound.arguments:
                    val = bound.arguments[param]
                    if not isinstance(val, expected):
                        raise TypeError(f"{param} must be {expected.__name__}, got {type(val).__name__}")
            return func(*args, **kwargs)
        return wrapper
    return decorator

@validate_types(name=str, age=int)
def create_user(name, age):
    return {"name": name, "age": age}

print(create_user("Alice", 30))
try:
    create_user("Bob", "thirty")  # raises TypeError
except TypeError as e:
    print(f"TypeError: {e}")

In [ ]:
# functools.lru_cache / functools.cache
from functools import lru_cache, cache

@lru_cache(maxsize=128)
def fib_lru(n):
    if n < 2: return n
    return fib_lru(n-1) + fib_lru(n-2)

@cache  # unbounded, Python 3.9+
def fib_cache(n):
    if n < 2: return n
    return fib_cache(n-1) + fib_cache(n-2)

start = time.perf_counter()
print(fib_lru(40))
print(f"With lru_cache: {time.perf_counter()-start:.4f}s")
print(fib_lru.cache_info())

## 3. Design Patterns

In [ ]:
# Singleton — thread-safe
import threading

class Singleton:
    _instance = None
    _lock = threading.Lock()

    def __new__(cls, *args, **kwargs):
        if not cls._instance:
            with cls._lock:
                if not cls._instance:  # double-checked locking
                    cls._instance = super().__new__(cls)
        return cls._instance

    def __init__(self, value=None):
        if not hasattr(self, "_initialized"):
            self.value = value
            self._initialized = True

s1 = Singleton(42)
s2 = Singleton(99)
print(s1 is s2)        # True — same instance
print(s1.value)        # 42 — first init wins

In [ ]:
# Factory Pattern
from abc import ABC, abstractmethod

class ModelBase(ABC):
    @abstractmethod
    def predict(self, x):
        pass

class LinearModel(ModelBase):
    def predict(self, x):
        return f"Linear prediction for {x}"

class TreeModel(ModelBase):
    def predict(self, x):
        return f"Tree prediction for {x}"

class NeuralModel(ModelBase):
    def predict(self, x):
        return f"Neural prediction for {x}"

class ModelFactory:
    _registry = {
        "linear": LinearModel,
        "tree": TreeModel,
        "neural": NeuralModel,
    }

    @classmethod
    def create(cls, model_type: str) -> ModelBase:
        if model_type not in cls._registry:
            raise ValueError(f"Unknown model: {model_type}. Choose from {list(cls._registry)}")
        return cls._registry[model_type]()

for mtype in ["linear", "tree", "neural"]:
    model = ModelFactory.create(mtype)
    print(model.predict([1, 2, 3]))

In [ ]:
# Observer Pattern — event system
from typing import Callable, List, Dict

class EventEmitter:
    def __init__(self):
        self._handlers: Dict[str, List[Callable]] = {}

    def on(self, event: str, handler: Callable):
        self._handlers.setdefault(event, []).append(handler)
        return self  # fluent interface

    def emit(self, event: str, *args, **kwargs):
        for handler in self._handlers.get(event, []):
            handler(*args, **kwargs)

    def off(self, event: str, handler: Callable):
        if event in self._handlers:
            self._handlers[event].remove(handler)

emitter = EventEmitter()
emitter.on("data", lambda x: print(f"Logger: {x}"))
emitter.on("data", lambda x: print(f"Processor: {x*2}"))
emitter.emit("data", 5)

In [ ]:
# Strategy Pattern
from typing import Protocol

class SortStrategy(Protocol):
    def sort(self, data: list) -> list: ...

class BubbleSort:
    def sort(self, data):
        arr = data[:]
        n = len(arr)
        for i in range(n):
            for j in range(n - i - 1):
                if arr[j] > arr[j+1]:
                    arr[j], arr[j+1] = arr[j+1], arr[j]
        return arr

class QuickSort:
    def sort(self, data):
        if len(data) <= 1:
            return data
        pivot = data[len(data)//2]
        left = [x for x in data if x < pivot]
        mid  = [x for x in data if x == pivot]
        right = [x for x in data if x > pivot]
        return self.sort(left) + mid + self.sort(right)

class Sorter:
    def __init__(self, strategy: SortStrategy):
        self._strategy = strategy
    def sort(self, data):
        return self._strategy.sort(data)

data = [5, 2, 8, 1, 9, 3]
for Strategy in [BubbleSort, QuickSort]:
    sorter = Sorter(Strategy())
    print(f"{Strategy.__name__}: {sorter.sort(data)}")

## 4. __slots__ & Descriptors

In [ ]:
import sys

class RegularPoint:
    def __init__(self, x, y):
        self.x = x
        self.y = y

class SlottedPoint:
    __slots__ = ("x", "y")
    def __init__(self, x, y):
        self.x = x
        self.y = y

reg = RegularPoint(1, 2)
slt = SlottedPoint(1, 2)
print(f"Regular: {sys.getsizeof(reg.__dict__)} bytes (dict overhead)")
print(f"Slotted: no __dict__ — more memory-efficient")

# __slots__ prevents arbitrary attribute assignment
reg.z = 3    # OK
try:
    slt.z = 3
except AttributeError as e:
    print(f"SlottedPoint error: {e}")

In [ ]:
# Descriptor Protocol
class Validated:
    """Descriptor that validates type and optional range."""
    def __init__(self, expected_type, min_val=None, max_val=None):
        self.expected_type = expected_type
        self.min_val = min_val
        self.max_val = max_val
        self.name = None

    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, obj, objtype=None):
        if obj is None:
            return self
        return getattr(obj, f"_{self.name}", None)

    def __set__(self, obj, value):
        if not isinstance(value, self.expected_type):
            raise TypeError(f"{self.name}: expected {self.expected_type.__name__}")
        if self.min_val is not None and value < self.min_val:
            raise ValueError(f"{self.name}: must be >= {self.min_val}")
        if self.max_val is not None and value > self.max_val:
            raise ValueError(f"{self.name}: must be <= {self.max_val}")
        setattr(obj, f"_{self.name}", value)

class Product:
    price = Validated(float, min_val=0.0)
    quantity = Validated(int, min_val=0, max_val=10000)
    name = Validated(str)

    def __init__(self, name, price, quantity):
        self.name = name
        self.price = price
        self.quantity = quantity

p = Product("Widget", 9.99, 100)
print(f"{p.name}: ${p.price} x {p.quantity}")

try:
    p.price = -1.0
except ValueError as e:
    print(f"ValueError: {e}")

## Interview Tips

- `asyncio` is for **I/O-bound** concurrency (network, disk). For **CPU-bound**: use `multiprocessing`.
- `asyncio.gather` vs `asyncio.create_task`: gather awaits all; create_task schedules but doesn't block.
- Decorator factories return a decorator (3 levels of nesting: factory → decorator → wrapper).
- Singleton is often an anti-pattern — consider dependency injection instead.
- `__slots__` shines in classes instantiated millions of times (ML feature vectors, graph nodes).
- Descriptors power `@property`, `@classmethod`, Django ORM fields — understand `__set_name__`.